# 4) Exploratory Data Analysis (EDA)

This notebook performs exploratory data analysis on the healthcare disease prediction dataset.

The objective is to understand:

- Dataset structure
- Missing and incomplete symptom values
- Disease distribution
- Symptom frequency and distribution
- Number of symptoms associated with each record
- Relationships between diseases and symptoms
- Duplicate records
- Categorical and numerical feature characteristics
- Important patterns and data-quality considerations

No permanent data cleaning or feature transformation is performed in this notebook.

In [ ]:
# Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import csv
from io import StringIO

In [ ]:
# Load dataset directly from GitHub

url = "https://raw.githubusercontent.com/vyasanbmathew2008/Team-6/main/healthcare-disease-prediction/dataset/healthcare_dataset.csv"

# Download CSV
response = requests.get(url)
response.raise_for_status()

# Read CSV without pandas' strict parser
reader = csv.reader(StringIO(response.text))
rows = list(reader)

# Separate header and data
header = rows[0]
data_rows = rows[1:]

# Find maximum number of fields
max_fields = max(len(row) for row in data_rows)

# Add column names if extra fields exist
while len(header) < max_fields:
    header.append(f"Symptom_{len(header)}")

# Make all rows the same length
fixed_rows = []

for row in data_rows:
    if len(row) < max_fields:
        row = row + [""] * (max_fields - len(row))
    elif len(row) > max_fields:
        row = row[:max_fields]

    fixed_rows.append(row)

# Create DataFrame
df = pd.DataFrame(fixed_rows, columns=header)

print("Dataset loaded successfully.")
print(f"Dataset shape: {df.shape}")

In [ ]:
# Display first five records

df.head()

In [ ]:
# Display dataset information
        
        "df.info()"

In [ ]:
# Display number of rows and columns
        
        "print(f\"Number of rows    : {df.shape[0]}\")"
        "print(f\"Number of columns : {df.shape[1]}\")"

## 1. Missing Value Analysis

The dataset contains disease and symptom fields. Some symptom fields may be empty because a record can contain fewer symptoms than the maximum number of symptom columns.

These empty symptom positions are examined as part of the exploratory analysis.

In [ ]:
# Identify disease and symptom columns

disease_column = "Disease"

symptom_columns = [
    column for column in df.columns
    if column.startswith("Symptom_")
]

print("Disease column:", disease_column)
print("Number of symptom columns:", len(symptom_columns))
print("Symptom columns:")
print(symptom_columns)

In [ ]:
# Calculate empty and missing values

missing_values = df.isnull().sum()

empty_values = (df == "").sum()

missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Empty Values": empty_values,
    "Total Missing/Empty": missing_values + empty_values
})

missing_summary

In [ ]:
# Calculate missing/empty percentages

missing_percentage = (
    missing_summary["Total Missing/Empty"] / len(df) * 100
).round(2)

missing_summary["Percentage"] = missing_percentage

missing_summary

In [ ]:
# Visualize empty symptom positions

symptom_empty_counts = (df[symptom_columns] == "").sum()

plt.figure(figsize=(12, 5))

sns.barplot(
    x=symptom_empty_counts.index,
    y=symptom_empty_counts.values
)

plt.title("Empty Values in Symptom Columns")
plt.xlabel("Symptom Column")
plt.ylabel("Number of Empty Values")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 2. Disease Distribution

The Disease column is the target variable for the disease prediction model.

The distribution of diseases is examined to identify the number of records available for each disease.

In [ ]:
# Disease class counts

disease_counts = df[disease_column].value_counts()

disease_counts

In [ ]:
# Disease class percentages

disease_distribution = pd.DataFrame({
    "Count": disease_counts,
    "Percentage": (
        disease_counts / len(df) * 100
    ).round(2)
})

disease_distribution

In [ ]:
# Plot disease distribution

plt.figure(figsize=(12, 8))

sns.countplot(
    data=df,
    y=disease_column,
    order=disease_counts.index
)

plt.title("Distribution of Diseases")
plt.xlabel("Number of Records")
plt.ylabel("Disease")
plt.tight_layout()
plt.show()

## 3. Symptom Frequency Analysis

All symptom columns are combined to determine how frequently individual symptoms occur throughout the dataset.

In [ ]:
# Combine all symptom values

all_symptoms = pd.concat(
        [df[column] for column in symptom_columns],
        ignore_index=True
    )

# Remove empty values
all_symptoms = all_symptoms[
        (all_symptoms != "") &
        (all_symptoms.notna())
    ]

symptom_counts = all_symptoms.value_counts()

print("Total symptom occurrences:", len(all_symptoms))
print("Unique symptoms:", all_symptoms.nunique())

In [ ]:
# Display symptom frequency table
        
        symptom_frequency = symptom_counts.reset_index()
        symptom_frequency.columns = ["Symptom", "Frequency"]
        
        symptom_frequency.head(30)

In [ ]:
# Plot the 20 most frequent symptoms
        
        top_symptoms = symptom_frequency.head(20)
        
        plt.figure(figsize=(10, 7))
        
        sns.barplot(
        data=top_symptoms,
        y="Symptom",
        x="Frequency"
        )
        
        plt.title("Top 20 Most Frequent Symptoms")
        plt.xlabel("Frequency")
        plt.ylabel("Symptom")
        plt.tight_layout()
        plt.show()

## 4. Number of Symptoms per Record

Each record may contain a different number of available symptoms. The number of non-empty symptoms in each record is calculated.

In [ ]:
# Count available symptoms for every record

df["Available_Symptom_Count"] = (
    df[symptom_columns]
    .replace("", np.nan)
    .notna()
    .sum(axis=1)
)

df["Available_Symptom_Count"].describe()

In [ ]:
# Display symptom count distribution

symptom_count_distribution = (
    df["Available_Symptom_Count"]
    .value_counts()
    .sort_index()
)

symptom_count_distribution

In [ ]:
# Plot number of symptoms per record

plt.figure(figsize=(9, 5))

sns.countplot(
    data=df,
    x="Available_Symptom_Count",
    order=sorted(df["Available_Symptom_Count"].unique())
)

plt.title("Number of Available Symptoms per Record")
plt.xlabel("Number of Symptoms")
plt.ylabel("Number of Records")
plt.tight_layout()
plt.show()

## 5. Disease vs Number of Symptoms

The average number of available symptoms for each disease is examined to identify differences in symptom coverage.

In [ ]:
# Average number of symptoms for each disease

disease_symptom_count = (
    df.groupby(disease_column)["Available_Symptom_Count"]
    .agg(["count", "mean", "min", "max"])
    .sort_values("mean", ascending=False)
)

disease_symptom_count

In [ ]:
# Plot average number of symptoms by disease

plt.figure(figsize=(12, 8))

sns.barplot(
    data=disease_symptom_count.reset_index(),
    y=disease_column,
    x="mean"
)

plt.title("Average Number of Symptoms per Disease")
plt.xlabel("Average Number of Symptoms")
plt.ylabel("Disease")
plt.tight_layout()
plt.show()

## 6. Disease-Symptom Relationship

The relationship between diseases and symptoms is explored using a disease-symptom frequency matrix.

In [ ]:
# Create disease-symptom frequency matrix

disease_symptom_matrix = pd.DataFrame(0, index=df[disease_column].unique(), columns=symptom_counts.index)

for _, row in df.iterrows():
    disease = row[disease_column]
    
    for column in symptom_columns:
        symptom = row[column]
        
        if symptom != "" and pd.notna(symptom):
            disease_symptom_matrix.loc[disease, symptom] += 1

disease_symptom_matrix.head()

In [ ]:
# Display the most common symptoms for each disease

for disease in disease_symptom_matrix.index:
    top_disease_symptoms = (
        disease_symptom_matrix.loc[disease]
        .sort_values(ascending=False)
        .head(5)
    )
    
    print("\n" + "=" * 60)
    print(disease)
    print("=" * 60)
    print(top_disease_symptoms)

## 7. Disease-Symptom Heatmap

A heatmap is created using the most frequent symptoms to visualize their occurrence across diseases.

In [ ]:
# Select top symptoms for heatmap

top_heatmap_symptoms = symptom_counts.head(20).index

heatmap_data = disease_symptom_matrix[
    top_heatmap_symptoms
]

plt.figure(figsize=(14, 12))

sns.heatmap(
    heatmap_data,
    cmap="YlGnBu",
    linewidths=0.2
)

plt.title("Disease-Symptom Frequency Heatmap")
plt.xlabel("Symptoms")
plt.ylabel("Disease")
plt.tight_layout()
plt.show()

## 8. Categorical Feature Analysis

The disease and symptom fields are categorical features. Their cardinality and frequency distributions are examined.

In [ ]:
# Identify categorical columns

categorical_columns = df.select_dtypes(
        include="object"
    ).columns.tolist()

print("Categorical/Text Columns:")

for column in categorical_columns:
    print("-", column)

print(f"\nTotal categorical/text columns: {len(categorical_columns)}")

In [ ]:
# Calculate categorical feature cardinality

categorical_summary = pd.DataFrame({
    "Column": categorical_columns,
    "Unique Values": [
        df[column].replace("", np.nan).nunique(dropna=True)
        for column in categorical_columns
    ],
    "Missing/Empty Values": [
        df[column].isna().sum() + (df[column] == "").sum()
        for column in categorical_columns
    ]
})

categorical_summary

## 9. Numerical Feature Analysis

The original healthcare disease dataset mainly contains categorical disease and symptom information.

The numerical columns are checked after loading the dataset. The generated `Available_Symptom_Count` is an analysis-only numerical feature and is not permanently added to the original dataset.

In [ ]:
# Identify original numerical columns

original_numerical_columns = [
    column for column in df.columns
    if column != "Available_Symptom_Count"
    and pd.api.types.is_numeric_dtype(df[column])
]

print("Original numerical columns:")

if original_numerical_columns:
    for column in original_numerical_columns:
        print("-", column)
else:
    print("No original numerical columns found.")

print(f"\nTotal original numerical columns: {len(original_numerical_columns)}")

In [ ]:
# Analyze the analysis-only numerical feature

df["Available_Symptom_Count"].describe()

In [ ]:
# Box plot of available symptom count

plt.figure(figsize=(8, 4))

sns.boxplot(
        data=df,
        x="Available_Symptom_Count"
    )

plt.title("Box Plot of Available Symptoms per Record")
plt.xlabel("Number of Available Symptoms")
plt.tight_layout()
plt.show()

## 10. Duplicate Record Analysis

Duplicate records are examined as part of the EDA process.

In [ ]:
# Count exact duplicate records

duplicate_count = df.drop(
        columns=["Available_Symptom_Count"]
    ).duplicated().sum()

duplicate_percentage = duplicate_count / len(df) * 100

print("Exact duplicate records:", duplicate_count)
print(f"Duplicate percentage: {duplicate_percentage:.2f}%")

## 11. Incomplete Record Analysis

Records containing fewer available symptoms than the maximum number of symptom fields are identified.

In [ ]:
# Identify incomplete records

incomplete_records = df[
        df["Available_Symptom_Count"] < len(symptom_columns)
    ]

complete_records = df[
        df["Available_Symptom_Count"] == len(symptom_columns)
    ]

print("Total records:", len(df))
print("Complete records:", len(complete_records))
print("Incomplete records:", len(incomplete_records))

print(
        f"Incomplete record percentage: "
        f"{len(incomplete_records) / len(df) * 100:.2f}%"
    )

In [ ]:
# Display sample incomplete records
        
        incomplete_records.head(20)

## 12. Data Quality Observations

The EDA highlights the following areas for later preprocessing:

- Empty symptom positions are present in some records.
- Different diseases may have different numbers of associated symptoms.
- Symptoms are categorical values and require appropriate encoding.
- Disease is the target variable for prediction.
- Duplicate records should be investigated before model training.
- Symptom naming inconsistencies such as similar terms may need standardization.
- The number of available symptoms varies between records.
- No permanent cleaning or transformation is performed in this notebook.

## 13. EDA Summary

In [ ]:
# Generate EDA summary

print("=" * 60)
print("EXPLORATORY DATA ANALYSIS SUMMARY")
print("=" * 60)
print(f"Total records                 : {len(df)}")
print(f"Total columns                 : {len(df.columns) - 1}")
print(f"Disease classes               : {df[disease_column].nunique()}")
print(f"Symptom columns               : {len(symptom_columns)}")
print(f"Unique symptoms               : {all_symptoms.nunique()}")
print(f"Total symptom occurrences     : {len(all_symptoms)}")
print(f"Complete records              : {len(complete_records)}")
print(f"Incomplete records            : {len(incomplete_records)}")
print(f"Exact duplicate records       : {duplicate_count}")
print(f"Average symptoms per record   : {df['Available_Symptom_Count'].mean():.2f}")
print(f"Minimum symptoms per record   : {df['Available_Symptom_Count'].min()}")
print(f"Maximum symptoms per record   : {df['Available_Symptom_Count'].max()}")
print("=" * 60)

# Conclusion

Exploratory Data Analysis was performed on the healthcare disease prediction dataset to understand its structure, disease distribution, symptom frequencies, symptom coverage, disease-symptom relationships, duplicate records, and incomplete records.

The analysis shows how disease and symptom information is distributed across the dataset and identifies important data-quality considerations that should be addressed during the preprocessing stage.

No permanent changes were made to the original dataset during this EDA notebook.